# Занятие 7. Частотный словарь, нормализация и простой спелчекер

**Цель практики:** собрать маленький словарь из открытых текстов, искусственно внести ошибки и проверить простой baseline исправления через edit distance.

Работает в бесплатном Colab на CPU.

In [ ]:
!pip -q install rapidfuzz pandas

In [ ]:
import os, re, json, textwrap, math, statistics, random, io
from pathlib import Path
import pandas as pd
import numpy as np
import requests

DATA_DIR = Path('/content/lowres_lab')
DATA_DIR.mkdir(exist_ok=True)

def show_df(df, n=10):
    display(df.head(n))

def save_artifact(name, obj):
    path = DATA_DIR / name
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    else:
        path.write_text(str(obj), encoding='utf-8')
    print('saved:', path)

from rapidfuzz import process, fuzz

## 1. Собираем слова из мини-корпуса

In [ ]:
LANG = 'udm'
API = f'https://{LANG}.wikipedia.org/w/api.php'

params = {
    'action': 'query',
    'generator': 'random',
    'grnnamespace': 0,
    'grnlimit': 20,
    'prop': 'extracts',
    'explaintext': 1,
    'format': 'json',
}
r = requests.get(API, params=params, timeout=30)
r.raise_for_status()
pages = r.json().get('query', {}).get('pages', {})
texts = [p.get('extract', '') for p in pages.values()]
tokens = []
for t in texts:
    tokens.extend(re.findall(r'[А-Яа-яЁёӐ-ӿ]{3,}', t.lower()))

freq = pd.Series(tokens).value_counts().reset_index()
freq.columns = ['word', 'count']
freq = freq[freq['count'] >= 1].head(300)
show_df(freq, 20)
save_artifact('lesson07_frequency_dictionary.csv', freq)

## 2. Делаем искусственные ошибки

In [ ]:
alphabet = sorted(set(''.join(freq['word'].head(100).tolist())))

def corrupt(word):
    if len(word) < 4:
        return word
    i = random.randrange(len(word))
    op = random.choice(['delete', 'swap', 'replace'])
    if op == 'delete':
        return word[:i] + word[i+1:]
    if op == 'swap' and i < len(word) - 1:
        return word[:i] + word[i+1] + word[i] + word[i+2:]
    return word[:i] + random.choice(alphabet or ['а']) + word[i+1:]

test_words = freq['word'].head(40).tolist()
typos = pd.DataFrame({'gold': test_words})
typos['typo'] = typos['gold'].apply(corrupt)
show_df(typos, 20)

## 3. Исправляем через ближайшее слово

In [ ]:
vocab = freq['word'].tolist()

def suggest(word, limit=3):
    return process.extract(word, vocab, scorer=fuzz.WRatio, limit=limit)

rows = []
for _, row in typos.iterrows():
    sugg = suggest(row['typo'])
    rows.append({
        'typo': row['typo'],
        'gold': row['gold'],
        'top1': sugg[0][0] if sugg else None,
        'top1_score': sugg[0][1] if sugg else None,
        'top3': [s[0] for s in sugg],
    })

eval_df = pd.DataFrame(rows)
eval_df['top1_correct'] = eval_df['top1'] == eval_df['gold']
eval_df['top3_correct'] = eval_df.apply(lambda r: r['gold'] in r['top3'], axis=1)
show_df(eval_df, 40)
print('top1 accuracy:', eval_df['top1_correct'].mean())
print('top3 accuracy:', eval_df['top3_correct'].mean())
save_artifact('lesson07_spellchecker_eval.csv', eval_df)

## 4. Где нормализация отличается от исправления

In [ ]:
normalization_questions = pd.DataFrame([
    {'case': 'вариант орфографии', 'should_fix?': 'не всегда', 'human_check': 'является ли форма допустимой нормой?'},
    {'case': 'OCR-ошибка', 'should_fix?': 'часто да', 'human_check': 'есть ли изображение-источник?'},
    {'case': 'диалектная форма', 'should_fix?': 'нет без решения сообщества', 'human_check': 'какой вариант нужен в корпусе?'},
    {'case': 'заимствование/имя', 'should_fix?': 'осторожно', 'human_check': 'это слово языка или шум?'},
])
show_df(normalization_questions)
save_artifact('lesson07_normalization_questions.csv', normalization_questions)

## Вопросы для отчёта

1. Какие ошибки baseline исправляет хорошо?
2. Какие “ошибки” могут оказаться нормальными вариантами?
3. Что нужно хранить в словаре: частоты, источники, варианты, комментарии?
4. Кто должен принимать решение о норме?